In [ ]:
# ensemble_pipeline_a100_yagpt.py

import os
import sys
import torch
import yaml
import argparse
import pandas as pd
import re
import subprocess
import git
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm import tqdm

# --- Проверка и импорт зависимостей ---
try:
    import wandb
    import evaluate
    import ctranslate2
    from datasets import load_dataset, Audio, Dataset
    from transformers import (
        WhisperForConditionalGeneration, WhisperProcessor,
        Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2SeqWithPadding,
        AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    )
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer
    from faster_whisper import WhisperModel as FasterWhisperModel
    from rover import Main as RoverMain
except ImportError as e:
    print(f"Ошибка: не найдена одна из необходимых библиотек. {e}")
    print("Пожалуйста, установите все зависимости: \n"
          "pip install torch pandas pyyaml gitpython tqdm evaluate scikit-learn wandb\n"
          "pip install datasets transformers accelerate ctranslate2 bitsandbytes scipy\n"
          "pip install faster-whisper py-rover peft trl")
    sys.exit(1)


# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---
# ... (load_config, normalize_text, run_rover_on_pair без изменений) ...
def load_config(config_path):
    with open(config_path, 'r', encoding='utf-8') as f: return yaml.safe_load(f)

def normalize_text(text):
    if not isinstance(text, str): return ""
    text = text.lower(); text = re.sub(r'[^\w\s]', '', text); text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_rover_on_pair(hyp1, hyp2, uid="utt"):
    # ... (код ROVER без изменений)
    hyp_files = []
    try:
        for i, hyp in enumerate([hyp1, hyp2]):
            filename = f"temp_hyp_{i}.txt"
            with open(filename, 'w', encoding='utf-8') as f: f.write(f"{hyp} ({uid})\n")
            hyp_files.append(filename)
        RoverMain.main(['-h', hyp_files[0], '-h', hyp_files[1], '-m', 'consensus', '-o', 'temp_out.txt'])
        with open('temp_out.txt', 'r', encoding='utf-8') as f: result_line = f.readline()
        return result_line.split(' (', 1)[0].strip()
    finally:
        for f in hyp_files + ['temp_out.txt']:
            if os.path.exists(f): os.remove(f)

# --- ЭТАП 1: ПОДГОТОВКА ДАННЫХ ---
def prepare_data(config):
    # ... (код из предыдущей версии без изменений) ...
    print("\n--- ЭТАП 1: ПОДГОТОВКА ДАННЫХ ---")
    # ... (логика остается прежней)
    return True

# --- НОВЫЙ ЭТАП: ПОДГОТОВКА ДАННЫХ ДЛЯ ДООБУЧЕНИЯ LLM ---
def prepare_llm_sft_data(config):
    print("\n--- ЭТАП 2: ПОДГОТОВКА ДАННЫХ ДЛЯ LLM FINE-TUNING ---")
    paths_cfg = config['paths']
    data_cfg = config['data_prep']
    llm_cfg = config['yandex_gpt_params']
    
    # Мы будем использовать обучающую выборку, чтобы научить LLM исправлять ошибки
    train_df = pd.read_json(Path(paths_cfg['prepared_data_dir']) / "whisper_train.jsonl", lines=True)

    # Загружаем базовую (недообученную) ASR-модель, чтобы сгенерировать "ошибочные" транскрипции
    print("Генерация 'ошибочных' транскрипций для SFT датасета...")
    whisper_base_model = FasterWhisperModel(config['whisper_params']['base_model'], device="cuda")
    
    sft_data = []
    prompt_template = llm_cfg['sft_prompt_template']

    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Создание SFT датасета"):
        audio_path = row['audio']
        correct_transcription = row['transcription']
        
        # Генерируем "плохую" транскрипцию
        segments, _ = whisper_base_model.transcribe(audio_path, language="ru")
        bad_transcription = "".join(s.text for s in segments).strip()

        # Формируем запись в формате instruction
        sft_data.append({
            "text": prompt_template.format(instruction=bad_transcription, response=correct_transcription)
        })

    # Сохраняем датасет
    sft_dataset = Dataset.from_list(sft_data)
    sft_dataset.save_to_disk(paths_cfg['llm_sft_data_path'])
    print(f"Датасет для SFT (Supervised Fine-Tuning) создан и сохранен в: {paths_cfg['llm_sft_data_path']}")
    return True


# --- ЭТАП 3: ДООБУЧЕНИЕ МОДЕЛЕЙ ---

def run_whisper_finetune(config):
    print("\n--- ЭТАП 3.1: ДООБУЧЕНИЕ И ОПТИМИЗАЦИЯ WHISPER ---")
    w_cfg = config['whisper_params']; paths_cfg = config['paths']
    
    os.environ["WANDB_PROJECT"] = w_cfg['wandb_project']
    
    dataset = load_dataset("json", data_files={'train': ..., 'test': ...}) # Загрузка данных
    # ... (остальной код подготовки датасета, как раньше) ...

    # ### ИЗМЕНЕНО: Используем bfloat16 для A100 и DeepSpeed/FSDP
    training_args = Seq2SeqTrainingArguments(
        output_dir=paths_cfg['whisper_finetuned_path'],
        report_to="wandb",
        bf16=True,  # bfloat16 - нативный и быстрый формат для A100
        # Для запуска на 2-х GPU, вы будете использовать `accelerate launch`.
        # Можно добавить конфиг DeepSpeed или FSDP для еще большей эффективности
        # deepspeed="path/to/ds_config.json",
        **w_cfg['training_args']
    )
    
    # ... (код Trainer'а, как раньше) ...
    # trainer.train()
    # ... (сохранение модели и конвертация в CTranslate2, как раньше) ...
    print("Обучение Whisper (пропущено для демонстрации).")


def run_silero_finetune(config):
    # ... (код без изменений) ...
    print("Обучение Silero (пропущено для демонстрации).")

# --- НОВЫЙ ЭТАП: ДООБУЧЕНИЕ YANDEXGPT ---
def run_yagpt_finetune(config):
    print("\n--- ЭТАП 3.3: ДООБУЧЕНИЕ YANDEXGPT ---")
    llm_cfg = config['yandex_gpt_params']
    paths_cfg = config['paths']

    # 1. Загрузка модели и токенизатора
    model_name = llm_cfg['base_model']
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # Конфиг для 4-битной квантизации для экономии памяти
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto", # Распределит модель по доступным GPU
        trust_remote_code=True
    )
    model.config.use_cache = False
    
    # 2. Настройка LoRA
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(**llm_cfg['lora_config'])
    model = get_peft_model(model, lora_config)
    
    # 3. Загрузка SFT датасета
    sft_dataset = Dataset.load_from_disk(paths_cfg['llm_sft_data_path'])
    
    # 4. Настройка и запуск обучения
    training_args = Seq2SeqTrainingArguments(
        output_dir=paths_cfg['llm_finetuned_path'],
        report_to="wandb",
        **llm_cfg['training_args']
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=sft_dataset,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=llm_cfg['max_seq_length'],
        tokenizer=tokenizer,
        args=training_args,
    )
    
    print("Запуск дообучения YandexGPT...")
    trainer.train()
    
    # 5. Сохранение адаптера и модели
    trainer.save_model(paths_cfg['llm_finetuned_path'])
    print(f"Дообученный LLM (адаптер) сохранен в: {paths_cfg['llm_finetuned_path']}")
    
# --- ЭТАП 4: ФИНАЛЬНАЯ ОЦЕНКА ---
def run_advanced_evaluation(config):
    # ... (Код почти без изменений, но LLM теперь дообученная) ...
    print("\n--- ЭТАП 4: ФИНАЛЬНАЯ ОЦЕНКА ---")
    # ... Загрузка ASR моделей ...
    
    # ### ИЗМЕНЕНО: Загрузка дообученной LLM с адаптером
    from peft import AutoPeftModelForCausalLM
    print("Загрузка дообученной модели LLM...")
    llm_model = AutoPeftModelForCausalLM.from_pretrained(
        config['paths']['llm_finetuned_path'],
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    llm_tokenizer = AutoTokenizer.from_pretrained(config['paths']['llm_finetuned_path'])
    
    # ... (Цикл инференса) ...
    # for row in test_df.iterrows():
        # ... (whisper, silero, rover инференс) ...
        
        # ### ИЗМЕНЕНО: Промпт для дообученной модели
        # prompt = llm_cfg['inference_prompt_template'].format(instruction=whisper_text_raw)
        # inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")
        # outputs = llm_model.generate(**inputs, max_new_tokens=...)
        # llm_corrected_text = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
        # ...
        
    # ... (Подсчет и логирование результатов) ...
    print("Финальная оценка (пропущено для демонстрации).")


# --- ГЛАВНАЯ ФУНКЦИЯ ---
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="config_a100_yagpt.yaml")
    args = parser.parse_args()
    config = load_config(args.config)
    
    p_ctrl = config['pipeline_control']
    if p_ctrl['run_data_prep']: prepare_data(config)
    if p_ctrl['run_llm_sft_data_prep']: prepare_llm_sft_data(config)
    
    # Можно запускать параллельно, если настроить CUDA_VISIBLE_DEVICES
    if p_ctrl['run_whisper_training']: run_whisper_finetune(config)
    if p_ctrl['run_silero_training']: run_silero_finetune(config)
    
    if p_ctrl['run_llm_finetuning']: run_yagpt_finetune(config)
    if p_ctrl['run_evaluation']: run_advanced_evaluation(config)
    
    print("\nПайплайн завершил работу.")

if __name__ == "__main__":
    main()